# Файнтюн Qwen3 на SEQ_CLS (36 классов)

Ноутбук — тонкая обёртка над `src.finetune.orchestrator.run_finetune`. Вся логика оркестрации (идемпотентность, чистка GPU между методами, OOM-handling, обновление сводной таблицы) — в ядре.

**Методы:**
- `lora` — Qwen3-14B + LoRA (bf16)
- `qlora` — Qwen3-32B + QLoRA (4bit NF4)
- `adalora` — Qwen3-32B + AdaLoRA (rank budget schedule)
- `tinylora` — Qwen3-32B + TinyLoRA (SVD-based, r=2, u=64)

**Идемпотентность:** при повторном запуске методы, у которых `run_key` уже есть в `results/finetune_results.csv`, пропускаются. Для пересчёта — `force=True`.

**Артефакты:**
- `Data/finetune_checkpoints/<run_key>/` — адаптер, токенизатор, `id2label.json`, `metadata.json`
- `results/finetune_results.csv` — агрегированные метрики (инкрементально)
- `results/preds_<run_key>.csv` — per-sample предсказания
- `results/all_methods_comparison.csv` — сводная таблица с остальными методами

___
## 1. Подготовка окружения (Colab)

Для локального запуска — замени блок на `PROJECT_ROOT = Path('/Users/.../code')` и закомментируй `drive.mount` + `pip install`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/VKR/code')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%cd {PROJECT_ROOT}

!pip install -q -r {PROJECT_ROOT}/requirements.txt 2>&1 | tail -5

## 2. GPU-профиль и импорт оркестратора

Одна строка конфигурации — как в `augmentation.ipynb`.

In [ ]:
from src.finetune.orchestrator import run_finetune

# === GPU ПРОФИЛЬ — МЕНЯЙ ЗДЕСЬ ===
GPU = 'A100_40'  # варианты: 'T4', 'L4', 'A100_40', 'A100_80', 'H100'

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}  ({vram:.1f} GB) | profile: {GPU}')
else:
    raise RuntimeError('CUDA недоступна. Смените Runtime на GPU.')

## 3. Выбор методов

`None` → полный прогон всех четырёх методов. Для частичного запуска — раскомментируй нужную строку.

In [ ]:
# Полный прогон всех методов:
METHODS = None

# Или конкретный набор:
# METHODS = ['lora']
# METHODS = ['qlora', 'adalora', 'tinylora']

# force=True → пересчитать уже готовые методы (иначе они пропускаются по run_key)
FORCE = False

## 4. Запуск файнтюна

Между методами оркестратор чистит GPU (`gc.collect()` + `torch.cuda.empty_cache()` + `synchronize()`) и ловит `OutOfMemoryError`, чтобы упавший метод не валил остальные.

In [ ]:
df = run_finetune(methods=METHODS, gpu=GPU, force=FORCE)
df

## 5. Сводная таблица со всеми методами

`results/all_methods_comparison.csv` содержит блоки `baseline` / `augmented` / `prompt` / `finetune` в одной схеме.

In [ ]:
import pandas as pd
pd.read_csv('results/all_methods_comparison.csv')